In [5]:
import pandas as pd
import numpy as np

# === Datei laden ===
instances = [
    "a3_o80_m10_an10_ar9_reduced",
    "a5_o96_m10_an10_ar10_reduced",
    "a10_o107_m5_an57_ar12",
    "a10_o114_m6_an57_ar11",
    "a10_o128_m6_an51_ar13",
    "a10_o144_m6_an53_ar12",
    "a15_o170_m9_an80_ar18",
    "a20_o236_m12_an106_ar24",
    "a25_o306_m13_an127_ar31",
    "a30_o355_m18_an148_ar42",
    "a40_o476_m22_an215_ar51",
    "a50_o578_m28_an276_ar66"]

for instance in instances:
    print(f"Processing instance: {instance}")
    df = pd.read_csv(f"{instance}/TPSA/ParetoFront.csv")  # ggf. Pfad anpassen

    # === Zielspalten definieren ===
    all_objectives = [
        "Driver Violation",
        "Commute Distance",
        "Transport Machines",
        "Transport Attachments",
        "Machines",
        "Workers",
        "Attachments"
    ]

    # === 1. Paretofront aus Transport Attachments & Attachments extrahieren ===
    def pareto_front_2d(points):
        points = np.array(points)
        is_efficient = np.ones(points.shape[0], dtype=bool)
        for i, c in enumerate(points):
            if is_efficient[i]:
                is_efficient[is_efficient] = (
                    np.any(points[is_efficient] < c, axis=1)
                    | np.all(points[is_efficient] == c, axis=1)
                )
                is_efficient[i] = True
        return points[is_efficient]

    pareto_points = pareto_front_2d(df[["Transport Attachments", "Attachments"]].values)
    df_pareto_attach = pd.DataFrame(
        pareto_points, columns=["Transport Attachments", "Attachments"]
    ).drop_duplicates()

    # === 2. Lösungen erweitern ===
    df_expanded = df.drop(columns=["Transport Attachments", "Attachments"]).merge(
        df_pareto_attach, how="cross"
    )

    # === 3. Finaler Paretofilter (alle Ziele) ===
    def pareto_filter_nd(df, objective_cols):
        values = df[objective_cols].values
        is_efficient = np.ones(values.shape[0], dtype=bool)
        for i, v in enumerate(values):
            if is_efficient[i]:
                is_efficient[is_efficient] = (
                    np.any(values[is_efficient] < v, axis=1)
                    | np.all(values[is_efficient] == v, axis=1)
                )
                is_efficient[i] = True
        return df[is_efficient].reset_index(drop=True)

    df_final_pareto = pareto_filter_nd(df_expanded, all_objectives)

    # === Insights ===
    unique_combos = df[["Transport Attachments", "Attachments"]].drop_duplicates()
    print("🔍 Insights zur Paretoanalyse\n" + "-"*40)
    print(f"📦 Ursprüngliche Lösungen:                   {len(df)}")
    print(f"🔢 Eindeutige TA/A-Kombinationen (vorher):   {len(unique_combos)}")
    print(f"🎯 Pareto-Kombinationen (2D, nicht-dominiert): {len(df_pareto_attach)}")
    print(f"🧩 Erweiterte Lösungskombis:                 {len(df_expanded)}")
    print(f"✅ Nicht-dominierte Endlösungen:             {len(df_final_pareto)}")
    print(f"❌ Entfernte/dazugenommene Lösungen:          {len(df_final_pareto) - len(df)}\n")

    print("📊 Pareto-Kombinationen (Anbaugeräte):")
    print(df_pareto_attach.sort_values(["Transport Attachments", "Attachments"]).to_string(index=False))

    # === Ergebnis speichern ===
    df_final_pareto.to_csv(f"{instance}/TPSA/ParetoFront_filtered.csv", index=False)
    print(f"\n💾 Datei gespeichert unter: ParetoFront_filtered.csv")

Processing instance: a3_o80_m10_an10_ar9_reduced
🔍 Insights zur Paretoanalyse
----------------------------------------
📦 Ursprüngliche Lösungen:                   108
🔢 Eindeutige TA/A-Kombinationen (vorher):   2
🎯 Pareto-Kombinationen (2D, nicht-dominiert): 1
🧩 Erweiterte Lösungskombis:                 108
✅ Nicht-dominierte Endlösungen:             84
❌ Entfernte/dazugenommene Lösungen:          -24

📊 Pareto-Kombinationen (Anbaugeräte):
 Transport Attachments  Attachments
                   0.0          2.0

💾 Datei gespeichert unter: ParetoFront_filtered.csv
Processing instance: a5_o96_m10_an10_ar10_reduced
🔍 Insights zur Paretoanalyse
----------------------------------------
📦 Ursprüngliche Lösungen:                   127
🔢 Eindeutige TA/A-Kombinationen (vorher):   1
🎯 Pareto-Kombinationen (2D, nicht-dominiert): 1
🧩 Erweiterte Lösungskombis:                 127
✅ Nicht-dominierte Endlösungen:             127
❌ Entfernte/dazugenommene Lösungen:          0

📊 Pareto-Kombinationen (A

In [8]:
import pandas as pd
import numpy as np

# === Instanzenliste ===
instances = [
    "a3_o80_m10_an10_ar9_reduced",
    "a5_o96_m10_an10_ar10_reduced",
    "a10_o107_m5_an57_ar12",
    "a10_o114_m6_an57_ar11",
    "a10_o128_m6_an51_ar13",
    "a10_o144_m6_an53_ar12",
    "a15_o170_m9_an80_ar18",
    "a20_o236_m12_an106_ar24",
    "a25_o306_m13_an127_ar31",
    "a30_o355_m18_an148_ar42",
    "a40_o476_m22_an215_ar51",
    "a50_o578_m28_an276_ar66",
    "RealLife_2024_7_1_2"
]

# === Zielspalten definieren ===
all_objectives = [
    "Driver Violation",
    "Commute Distance",
    "Transport Machines",
    "Transport Attachments",
    "Machines",
    "Workers",
    "Attachments"
]

# === 2D Paretofilter (nur 2 Ziele)
def pareto_front_2d(points):
    points = np.array(points)
    is_efficient = np.ones(points.shape[0], dtype=bool)
    for i, c in enumerate(points):
        if is_efficient[i]:
            is_efficient[is_efficient] = (
                np.any(points[is_efficient] < c, axis=1)
                | np.all(points[is_efficient] == c, axis=1)
            )
            is_efficient[i] = True
    return points[is_efficient]

# === ND Paretofilter (alle Ziele)
def pareto_filter_nd(df, objective_cols):
    values = df[objective_cols].values
    is_efficient = np.ones(values.shape[0], dtype=bool)
    for i, v in enumerate(values):
        if is_efficient[i]:
            is_efficient[is_efficient] = (
                np.any(values[is_efficient] < v, axis=1)
                | np.all(values[is_efficient] == v, axis=1)
            )
            is_efficient[i] = True
    return df[is_efficient].reset_index(drop=True)

# === Ergebnisse pro Instanz sammeln
summary_rows = []

# === Hauptloop
for instance in instances:
    print(f"🔄 Processing instance: {instance}")
    df = pd.read_csv(f"{instance}/TPSA/ParetoFront.csv")

    # --- Paretofront 2D aus Attachments ---
    pareto_points = pareto_front_2d(df[["Transport Attachments", "Attachments"]].values)
    df_pareto_attach = pd.DataFrame(pareto_points, columns=["Transport Attachments", "Attachments"]).drop_duplicates()

    # --- Lösungen erweitern ---
    df_expanded = df.drop(columns=["Transport Attachments", "Attachments"]).merge(df_pareto_attach, how="cross")

    # --- ND Paretofilter auf gesamte Lösung ---
    df_final_pareto = pareto_filter_nd(df_expanded, all_objectives)

    # --- Einzelwerte für Zusammenfassung ---
    original = len(df)
    unique_attach_combos = len(df[["Transport Attachments", "Attachments"]].drop_duplicates())
    pareto_2d = len(df_pareto_attach)
    expanded = len(df_expanded)
    final = len(df_final_pareto)
    delta = final - original

    summary_rows.append({
        "Instance": instance,
        "Original Solutions": original,
        "Unique (TA, A) Combos": unique_attach_combos,
        "Pareto (TA, A) Combos": pareto_2d,
        "Expanded Solutions": expanded,
        "Final Pareto Solutions": final,
        "Δ Solutions": delta
    })

    # --- Speichern
    #df_final_pareto.to_csv(f"{instance}/TPSA/ParetoFront_filtered.csv", index=False)
    #print(f"✅ Saved filtered Pareto front: {len(df_final_pareto)} solutions\n")

# === Zusammenfassung anzeigen
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

🔄 Processing instance: a3_o80_m10_an10_ar9_reduced
🔄 Processing instance: a5_o96_m10_an10_ar10_reduced
🔄 Processing instance: a10_o107_m5_an57_ar12
🔄 Processing instance: a10_o114_m6_an57_ar11
🔄 Processing instance: a10_o128_m6_an51_ar13
🔄 Processing instance: a10_o144_m6_an53_ar12
🔄 Processing instance: a15_o170_m9_an80_ar18
🔄 Processing instance: a20_o236_m12_an106_ar24
🔄 Processing instance: a25_o306_m13_an127_ar31
🔄 Processing instance: a30_o355_m18_an148_ar42
🔄 Processing instance: a40_o476_m22_an215_ar51
🔄 Processing instance: a50_o578_m28_an276_ar66
🔄 Processing instance: RealLife_2024_7_1_2


,Instance,Original Solutions,"Unique (TA, A) Combos","Pareto (TA, A) Combos",Expanded Solutions,Final Pareto Solutions,Δ Solutions
0,a3_o80_m10_an10_ar9_reduced,108,2,1,108,84,-24
1,a5_o96_m10_an10_ar10_reduced,127,1,1,127,127,0
2,a10_o107_m5_an57_ar12,248,1,1,248,247,-1
3,a10_o114_m6_an57_ar11,295,19,2,590,148,-147
4,a10_o128_m6_an51_ar13,696,2,2,1392,994,298
5,a10_o144_m6_an53_ar12,311,1,1,311,311,0
6,a15_o170_m9_an80_ar18,1149,162,4,4596,676,-473
7,a20_o236_m12_an106_ar24,672,143,5,3360,135,-537
8,a25_o306_m13_an127_ar31,962,387,4,3848,384,-578
9,a30_o355_m18_an148_ar42,1260,165,5,6300,2320,1060
